## 1. Motivation

### What is our dataset?
We use four different datasets from Eurostat, which is the Statistical Office of the European Union. All datasets belong to the **"Quality of life"** section.  
More specifically, there is one dataset covering *Mean and median income by age and sex*, while the other three are part of the **Health** subsection: 
- *Daily smokers of cigarettes by sex, age and income quintile*, 
- *Frequency of heavy episodic drinking by sex, age and income quintile* and 
- *Current depressive symptoms by sex, age and income quintile*.

The data is aggregated at the population level and spans multiple European countries, enabling cross-country comparisons.

### Why did we choose these particular datasets?
We chose these datasets because they allow us to explore the relationship between socioeconomic status (income) and health-related behaviors and outcomes. Income is a well-known determinant of health, and combining it with indicators such as smoking, alcohol consumption, and depressive symptoms enables a more comprehensive analysis of inequality in quality of life.
Additionally, the datasets share common variables (age, sex, income quintile), which makes them suitable for integration and comparative analysis.

### What was our goal for the end user's experience?
Our goal was to create an intuitive and informative data story that allows users to:
- Explore how income relates to different health outcomes
- Compare patterns across demographic groups (age, sex)
- Identify inequalities between different countries and income groups

We aimed to present the results in a clear and engaging way by using interactive visualizations that allow users to actively explore the data. For example, we incorporated clickable maps that enable users to navigate between countries and examine patterns in more detail, helping them develop a more intuitive understanding of the underlying trends.

## 2. Basic stats

### Data cleaning and preprocessing

In [35]:
import pandas as pd

# Load data
income_data = pd.read_csv('data/mean_and_median_income_by_age_and_sex.csv')
# Show category names
print(income_data.columns)

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'age', 'Age class', 'sex', 'Sex', 'indic_il',
       'Income and living conditions indicator', 'unit', 'Unit of measure',
       'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'Time',
       'OBS_VALUE', 'Observation value', 'OBS_FLAG',
       'Observation status (Flag) V2 structure', 'CONF_STATUS',
       'Confidentiality status (flag)'],
      dtype='str')


In [ ]:
import re

import numpy as np
import pandas as pd

In [38]:
def load_data(path: str) -> pd.DataFrame:
    """Load CSV into a DataFrame with sensible defaults."""
    df = pd.read_csv(path, low_memory=False)
    return df

def _clean_column_name(name: str) -> str:
    name = name.strip()
    name = name.replace("/", "_")
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^0-9a-zA-Z_]+", "", name)
    return name.lower()

def inspect(df: pd.DataFrame) -> None:
    """Print basic dataset diagnostics."""
    print("Shape:", df.shape)
    print("Columns:")
    for col in df.columns:
        non_null = df[col].notna().sum()
        pct_missing = 100 * (1 - non_null / len(df))
        print(f" - {col}: dtype={df[col].dtype}, missing={pct_missing:.1f}%")

def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Perform generic cleaning steps and return a cleaned copy.

    Steps performed:
    - Normalize column names
    - Make column names unique to avoid indexing issues
    - Replace common missing-value tokens with np.nan
    - Strip strings and convert obvious numeric columns to numeric dtypes
    - Drop fully-empty columns and exact duplicate rows
    - Set likely categorical columns to category dtype (geo, sex, age)
    """
    df = df.copy()

    # Normalize column names FIRST
    df.columns = [_clean_column_name(c) for c in df.columns]
    
    # Make column names unique by appending index if duplicates exist
    cols = df.columns.tolist()
    seen = {}
    new_cols = []
    for col in cols:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols

    # Replace common tokens with NaN
    df.replace(list({""}), np.nan, inplace=True)

    # Strip whitespace from object columns and handle "nan" strings
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace({"nan": np.nan})

    # Drop fully-empty columns
    df.dropna(axis=1, how="all", inplace=True)

    # Drop exact duplicates
    df.drop_duplicates(inplace=True)

    # Try to coerce numeric-like columns
    for col in df.columns:
        if df[col].dtype == object:
            sample = df[col].dropna().head(200).astype(str)
            # Remove thousands separators and convert commas to dots if present
            cleaned = sample.str.replace(r"\s+", "", regex=True).str.replace(",", ".")
            # Remove any non-numeric trailing characters
            cleaned = cleaned.str.replace(r"[^0-9eE+\-\.]", "", regex=True)
            num_parsable = pd.to_numeric(cleaned, errors="coerce").notna().sum()
            if len(sample) > 0 and (num_parsable / len(sample)) > 0.6:
                # majority numeric-like -> coerce whole column
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.replace(r"\s+", "", regex=True)
                    .str.replace(",", ".")
                    .str.replace(r"[^0-9eE+\-\.]", "", regex=True)
                )
                df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def save_processed(df: pd.DataFrame, path: str) -> None:
    """Save processed DataFrame to CSV (index=False)."""
    df.to_csv(path, index=False)

### Preprocessing for income data

In [39]:
# Load raw data and inspect
df_income_raw = load_data('data/mean_and_median_income_by_age_and_sex.csv')
print("=== Before cleaning ===")
inspect(df_income_raw)

=== Before cleaning ===
Shape: (329820, 23)
Columns:
 - STRUCTURE: dtype=str, missing=0.0%
 - STRUCTURE_ID: dtype=str, missing=0.0%
 - STRUCTURE_NAME: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - Time frequency: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - Age class: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - Sex: dtype=str, missing=0.0%
 - indic_il: dtype=str, missing=0.0%
 - Income and living conditions indicator: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - Unit of measure: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - Geopolitical entity (reporting): dtype=str, missing=0.0%
 - TIME_PERIOD: dtype=int64, missing=0.0%
 - Time: dtype=float64, missing=100.0%
 - OBS_VALUE: dtype=int64, missing=0.0%
 - Observation value: dtype=float64, missing=100.0%
 - OBS_FLAG: dtype=str, missing=92.5%
 - Observation status (Flag) V2 structure: dtype=str, missing=92.5%
 - CONF_STATUS: dtype=float64, missing=100.0%
 - Confidentialit

In [40]:
# Clean the data
df_income_cleaned = clean(df_income_raw)
print("=== After cleaning ===")
inspect(df_income_cleaned)

=== After cleaning ===
Shape: (329820, 19)
Columns:
 - structure: dtype=str, missing=0.0%
 - structure_id: dtype=str, missing=0.0%
 - structure_name: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - time_frequency: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - age_class: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - sex_1: dtype=str, missing=0.0%
 - indic_il: dtype=str, missing=0.0%
 - income_and_living_conditions_indicator: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - unit_of_measure: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - geopolitical_entity_reporting: dtype=str, missing=0.0%
 - time_period: dtype=int64, missing=0.0%
 - obs_value: dtype=int64, missing=0.0%
 - obs_flag: dtype=str, missing=92.5%
 - observation_status_flag_v2_structure: dtype=str, missing=92.5%


In [41]:
# Save the processed data
output_path = 'data/mean_and_median_income_by_age_and_sex.cleaned.csv'
save_processed(df_income_cleaned, output_path)
print(f"Cleaned data saved to {output_path}")
print(f"\nFinal shape: {df_income_cleaned.shape}")

Cleaned data saved to data/mean_and_median_income_by_age_and_sex.cleaned.csv

Final shape: (329820, 19)


### Preprocessing for smoker data

In [25]:
# Load raw data and inspect
df_smokers_raw = load_data('data/daily_smokers_of_cigarettes_by_sex_age_and_income_quintile.csv')
print("=== Before cleaning ===")
inspect(df_smokers_raw)

=== Before cleaning ===
Shape: (66582, 25)
Columns:
 - STRUCTURE: dtype=str, missing=0.0%
 - STRUCTURE_ID: dtype=str, missing=0.0%
 - STRUCTURE_NAME: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - Time frequency: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - Unit of measure: dtype=str, missing=0.0%
 - smoking: dtype=str, missing=0.0%
 - Smoking behaviour: dtype=str, missing=0.0%
 - quant_inc: dtype=str, missing=0.0%
 - Income quantile: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - Sex: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - Age class: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - Geopolitical entity (reporting): dtype=str, missing=0.0%
 - TIME_PERIOD: dtype=int64, missing=0.0%
 - Time: dtype=float64, missing=100.0%
 - OBS_VALUE: dtype=float64, missing=3.0%
 - Observation value: dtype=float64, missing=100.0%
 - OBS_FLAG: dtype=str, missing=87.0%
 - Observation status (Flag) V2 structure: dtype=str, missing=87.0%
 -

In [26]:
# Clean the data
df_smokers_cleaned = clean(df_smokers_raw)
print("=== After cleaning ===")
inspect(df_smokers_cleaned)

=== After cleaning ===
Shape: (66582, 21)
Columns:
 - structure: dtype=str, missing=0.0%
 - structure_id: dtype=str, missing=0.0%
 - structure_name: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - time_frequency: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - unit_of_measure: dtype=str, missing=0.0%
 - smoking: dtype=str, missing=0.0%
 - smoking_behaviour: dtype=str, missing=0.0%
 - quant_inc: dtype=str, missing=0.0%
 - income_quantile: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - sex_1: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - age_class: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - geopolitical_entity_reporting: dtype=str, missing=0.0%
 - time_period: dtype=int64, missing=0.0%
 - obs_value: dtype=float64, missing=3.0%
 - obs_flag: dtype=str, missing=87.0%
 - observation_status_flag_v2_structure: dtype=str, missing=87.0%


In [28]:
# Save the processed data
output_path = 'data/daily_smokers_of_cigarettes_by_sex_age_and_income_quintile.cleaned.csv'
save_processed(df_smokers_cleaned, output_path)
print(f"Cleaned data saved to {output_path}")
print(f"\nFinal shape: {df_smokers_cleaned.shape}")

Cleaned data saved to data/daily_smokers_of_cigarettes_by_sex_age_and_income_quintile.cleaned.csv

Final shape: (66582, 21)


### Preprocessing on depression data

In [29]:
# Load raw data and inspect
df_depressive_raw = load_data('data/current_depressive_symptoms_by_sex_age_and_income_quintile.csv')
print("=== Before cleaning ===")
inspect(df_depressive_raw)

=== Before cleaning ===
Shape: (52272, 25)
Columns:
 - STRUCTURE: dtype=str, missing=0.0%
 - STRUCTURE_ID: dtype=str, missing=0.0%
 - STRUCTURE_NAME: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - Time frequency: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - Unit of measure: dtype=str, missing=0.0%
 - hlth_pb: dtype=str, missing=0.0%
 - Health problems: dtype=str, missing=0.0%
 - quant_inc: dtype=str, missing=0.0%
 - Income quantile: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - Sex: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - Age class: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - Geopolitical entity (reporting): dtype=str, missing=0.0%
 - TIME_PERIOD: dtype=int64, missing=0.0%
 - Time: dtype=float64, missing=100.0%
 - OBS_VALUE: dtype=float64, missing=2.5%
 - Observation value: dtype=float64, missing=100.0%
 - OBS_FLAG: dtype=str, missing=88.2%
 - Observation status (Flag) V2 structure: dtype=str, missing=88.2%
 - C

In [30]:
# Clean the data
df_depressive_cleaned = clean(df_depressive_raw)
print("=== After cleaning ===")
inspect(df_depressive_cleaned)

=== After cleaning ===
Shape: (52272, 21)
Columns:
 - structure: dtype=str, missing=0.0%
 - structure_id: dtype=str, missing=0.0%
 - structure_name: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - time_frequency: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - unit_of_measure: dtype=str, missing=0.0%
 - hlth_pb: dtype=str, missing=0.0%
 - health_problems: dtype=str, missing=0.0%
 - quant_inc: dtype=str, missing=0.0%
 - income_quantile: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - sex_1: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - age_class: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - geopolitical_entity_reporting: dtype=str, missing=0.0%
 - time_period: dtype=int64, missing=0.0%
 - obs_value: dtype=float64, missing=2.5%
 - obs_flag: dtype=str, missing=88.2%
 - observation_status_flag_v2_structure: dtype=str, missing=88.2%


In [31]:
# Save the processed data
output_path = 'data/current_depressive_symptoms_by_sex_age_and_income_quintile.cleaned.csv'
save_processed(df_depressive_cleaned, output_path)
print(f"Cleaned data saved to {output_path}")
print(f"\nFinal shape: {df_depressive_cleaned.shape}")

Cleaned data saved to data/current_depressive_symptoms_by_sex_age_and_income_quintile.cleaned.csv

Final shape: (52272, 21)


### Preprocessing on drinking data

In [32]:
# Load raw data and inspect
df_drinking_raw = load_data('data/frequency_of_heavy_episodic_drinking_by_sex_age_and_income_quintile.csv')
print("=== Before cleaning ===")
inspect(df_drinking_raw)

=== Before cleaning ===
Shape: (84600, 25)
Columns:
 - STRUCTURE: dtype=str, missing=0.0%
 - STRUCTURE_ID: dtype=str, missing=0.0%
 - STRUCTURE_NAME: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - Time frequency: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - Unit of measure: dtype=str, missing=0.0%
 - quant_inc: dtype=str, missing=0.0%
 - Income quantile: dtype=str, missing=0.0%
 - frequenc: dtype=str, missing=0.0%
 - Frequency: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - Sex: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - Age class: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - Geopolitical entity (reporting): dtype=str, missing=0.0%
 - TIME_PERIOD: dtype=int64, missing=0.0%
 - Time: dtype=float64, missing=100.0%
 - OBS_VALUE: dtype=float64, missing=3.4%
 - Observation value: dtype=float64, missing=100.0%
 - OBS_FLAG: dtype=str, missing=81.4%
 - Observation status (Flag) V2 structure: dtype=str, missing=81.4%
 - CONF_S

In [33]:
# Clean the data
df_drinking_cleaned = clean(df_drinking_raw)
print("=== After cleaning ===")
inspect(df_drinking_cleaned)

=== After cleaning ===
Shape: (84600, 21)
Columns:
 - structure: dtype=str, missing=0.0%
 - structure_id: dtype=str, missing=0.0%
 - structure_name: dtype=str, missing=0.0%
 - freq: dtype=str, missing=0.0%
 - time_frequency: dtype=str, missing=0.0%
 - unit: dtype=str, missing=0.0%
 - unit_of_measure: dtype=str, missing=0.0%
 - quant_inc: dtype=str, missing=0.0%
 - income_quantile: dtype=str, missing=0.0%
 - frequenc: dtype=str, missing=0.0%
 - frequency: dtype=str, missing=0.0%
 - sex: dtype=str, missing=0.0%
 - sex_1: dtype=str, missing=0.0%
 - age: dtype=str, missing=0.0%
 - age_class: dtype=str, missing=0.0%
 - geo: dtype=str, missing=0.0%
 - geopolitical_entity_reporting: dtype=str, missing=0.0%
 - time_period: dtype=int64, missing=0.0%
 - obs_value: dtype=float64, missing=3.4%
 - obs_flag: dtype=str, missing=81.4%
 - observation_status_flag_v2_structure: dtype=str, missing=81.4%


In [34]:
# Save the processed data
output_path = 'data/frequency_of_heavy_episodic_drinking_by_sex_age_and_income_quintile.cleaned.csv'
save_processed(df_drinking_cleaned, output_path)
print(f"Cleaned data saved to {output_path}")
print(f"\nFinal shape: {df_drinking_cleaned.shape}")

Cleaned data saved to data/frequency_of_heavy_episodic_drinking_by_sex_age_and_income_quintile.cleaned.csv

Final shape: (84600, 21)


### Dataset stats (key points/ plots from our exploratory data analysis)

## 3. Data Analysis

### Describe data analysis and what we have learned about the dataset

### ML?

## 4. Genre

### Which tools did we use from each of the categories of Visual Narrative (Figure 7 in Segal and Heer) and why?

### Which toold did we use from each of the categories of Narrative Structure (Figure 7 in Segal and Heer) and why?

## 5. Visualizations

### Explanation of visulaizations that we chose.

### Why are they right for the story that we want to tell?

## 6. Discussion

### What went well?

### What is still missing? What could be improved? Why?

## 7. Contributions

## 8. References